In [3]:
%conda install PyVCF

Solving environment: / 
The environment is inconsistent, please check the package plan carefully
The following packages are causing the inconsistency:

  - conda-forge/osx-64::gym-all==0.21.0=py39h6e9494a_2
  - conda-forge/osx-64::gym-toy_text==0.21.0=py39h6e9494a_2
done


==> WARNING: A newer version of conda exists. <==
  current version: 22.9.0
  latest version: 24.3.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /Users/fionachow/opt/anaconda3/envs/rl

  added / updated specs:
    - pyvcf


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2024.2.2   |       h8857fd0_0         152 KB  conda-forge
    openssl-3.2.1              |       hd75f5a5_1         2.4 MB  conda-forge
    pyvcf-0.6.8                |py39h6e9494a_1003          69 KB  conda-forge
    scipy-1.12.0               |   py39h

In [4]:
import io
import os
import pandas as pd
import re
import vcf

In [18]:
def read_vcf(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
               'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    ).rename(columns={'#CHROM': 'CHROM'})

def extract_af(info):
    match = re.search(r'AF=([\d\.]+)', info)
    return match.group(1) if match else None

def filter_vcf(vcf_file):
    vcf_reader = vcf.Reader(open(vcf_file, 'r'))

    for record in vcf_reader:
        if 'AF' in record.INFO and record.INFO['AF'] > 0.5:
            print(record)

In [6]:
df = read_vcf('/Users/fionachow/Documents/ERR3240115_rDNA.vcf') #vcf file from original variant calling of 1 individual

df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1"
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1"
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10"
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10"
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE..."
...,...,...,...,...,...,...,...,...
374,Human,13219,.,C,CG,43670,PASS,"DP=2182;AF=0.944088;SB=1;DP4=1,152,29,2031;IND..."
375,Human,13235,.,GC,G,35659,PASS,"DP=1803;AF=0.953411;SB=22;DP4=5,91,18,1701;IND..."
376,Human,13278,.,G,GGC,13133,PASS,"DP=1033;AF=0.909971;SB=0;DP4=0,103,2,938;INDEL..."
377,Human,13280,.,G,C,13572,PASS,"DP=1017;AF=1.000000;SB=0;DP4=0,0,2,1015"


In [7]:
#extracting AF into a seperate column
df['AF'] = df['INFO'].apply(extract_af)

df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1",1.000000
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1",1.000000
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10",0.996952
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10",0.999624
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
...,...,...,...,...,...,...,...,...,...
374,Human,13219,.,C,CG,43670,PASS,"DP=2182;AF=0.944088;SB=1;DP4=1,152,29,2031;IND...",0.944088
375,Human,13235,.,GC,G,35659,PASS,"DP=1803;AF=0.953411;SB=22;DP4=5,91,18,1701;IND...",0.953411
376,Human,13278,.,G,GGC,13133,PASS,"DP=1033;AF=0.909971;SB=0;DP4=0,103,2,938;INDEL...",0.909971
377,Human,13280,.,G,C,13572,PASS,"DP=1017;AF=1.000000;SB=0;DP4=0,0,2,1015",1.000000


In [14]:
type(df.loc[2, 'AF']) #string type

str

In [16]:
df['AF'] = df['AF'].astype(float) #converting format to float

type(df.loc[2, 'AF'])

numpy.float64

In [18]:
filtered_df = df[df['AF'] > 0.5] #filtering for AF threshold >0.5; extracting majority alleles at each position from variant calling

filtered_df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1",1.000000
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1",1.000000
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10",0.996952
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10",0.999624
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
...,...,...,...,...,...,...,...,...,...
374,Human,13219,.,C,CG,43670,PASS,"DP=2182;AF=0.944088;SB=1;DP4=1,152,29,2031;IND...",0.944088
375,Human,13235,.,GC,G,35659,PASS,"DP=1803;AF=0.953411;SB=22;DP4=5,91,18,1701;IND...",0.953411
376,Human,13278,.,G,GGC,13133,PASS,"DP=1033;AF=0.909971;SB=0;DP4=0,103,2,938;INDEL...",0.909971
377,Human,13280,.,G,C,13572,PASS,"DP=1017;AF=1.000000;SB=0;DP4=0,0,2,1015",1.000000


In [19]:
#alt way to read vcf files filtered for variant positions
filter_vcf('/Users/fionachow/Documents/ERR3240115_rDNA.vcf') #similar outputs to the pandas filter

Record(CHROM=Human, POS=31, REF=T, ALT=[C])
Record(CHROM=Human, POS=32, REF=C, ALT=[T])
Record(CHROM=Human, POS=73, REF=C, ALT=[A])
Record(CHROM=Human, POS=74, REF=A, ALT=[C])
Record(CHROM=Human, POS=80, REF=A, ALT=[AC])
Record(CHROM=Human, POS=103, REF=GC, ALT=[G])
Record(CHROM=Human, POS=121, REF=C, ALT=[G])
Record(CHROM=Human, POS=237, REF=G, ALT=[GC])
Record(CHROM=Human, POS=270, REF=C, ALT=[CG])
Record(CHROM=Human, POS=275, REF=C, ALT=[CG])
Record(CHROM=Human, POS=284, REF=A, ALT=[AC])
Record(CHROM=Human, POS=363, REF=G, ALT=[GT])
Record(CHROM=Human, POS=386, REF=T, ALT=[TGGCC])
Record(CHROM=Human, POS=419, REF=C, ALT=[CGT])
Record(CHROM=Human, POS=517, REF=G, ALT=[GC])
Record(CHROM=Human, POS=522, REF=C, ALT=[CG])
Record(CHROM=Human, POS=524, REF=C, ALT=[G])
Record(CHROM=Human, POS=537, REF=TG, ALT=[T])
Record(CHROM=Human, POS=560, REF=C, ALT=[G])
Record(CHROM=Human, POS=561, REF=G, ALT=[GT])
Record(CHROM=Human, POS=623, REF=C, ALT=[CT])
Record(CHROM=Human, POS=626, REF=T, ALT=[G

In [17]:
mpileup_file_path = '/Users/fionachow/Documents/ERR3240115_rDNA.mpileup' #extracting reference alleles for all Positions

mpileup_df = pd.read_csv(mpileup_file_path, sep='\t', header=None, usecols=[0, 1, 2], names=['CHROM', 'POS', 'REF'])

mpileup_df.head(32)

,CHROM,POS,REF
0,Human,1,G
1,Human,2,C
2,Human,3,T
3,Human,4,G
4,Human,5,A
5,Human,6,C
6,Human,7,A
7,Human,8,C
8,Human,9,G
9,Human,10,C
